# GAVE2 CMRRWNet First on Google Colab

This notebook runs only the CMRRWNet branch first. That gives you one clean baseline output to submit/test before spending time on SAM3, YOLO-native, or ensembling.

Outputs will be saved to:

```text
MyDrive/MICCAI2026/submissions/cmrrwnet/team_id/
  Task1/*.png
  Task2/*.png
  Task3/*.txt
```

The pipeline keeps the full native image canvas: `1536 x 1024`, no crop, no resize.

## 0. Before You Start

In Colab, choose:

`Runtime` -> `Change runtime type` -> `GPU`

A100 is recommended for full training. L4/T4 is fine for smoke tests.

Your Drive should contain either one project zip or the loose folders:

```text
MyDrive/MICCAI2026/
  miccai.zip          # preferred: contains GAVE2_preliminary, experiments, and optionally knowledge_base
  # or:
  GAVE2_preliminary.zip # dataset-only zip, plus experiments folder below
  experiments/          # needed if not inside the zip
  knowledge_base/       # optional, used for official CMRRWNet fallback if present
```

In [ ]:
#@title Configuration
from pathlib import Path

PROJECT_NAME = "MICCAI2026"
TEAM_ID = "team_id"  # change this to your official team ID before final zip
DRIVE_PROJECT = Path("/content/drive/MyDrive") / PROJECT_NAME
WORKDIR = Path("/content") / PROJECT_NAME
DATA_ROOT = WORKDIR / "GAVE2_preliminary"
RUN_DIR = DRIVE_PROJECT / "runs" / "gave2_cmrrwnet_first"
SUBMISSION_ROOT = DRIVE_PROJECT / "submissions"

# A100-friendly defaults.
BASE_CHANNELS = 64
EPOCHS_FULL = 150  # max cap; early stopping usually ends earlier
FOLDS = 5
BATCH_SIZE = 1
GRAD_ACCUM = 1
EARLY_STOPPING_PATIENCE = 25
EARLY_STOPPING_MIN_DELTA = 1e-4
WORKERS = 2
AMP = "bf16"
PREPROCESS = "gray_clahe"
LOSS_MODE = "official_bce3"
BRANCH = "cmrrwnet"


# If you uploaded one zip to MyDrive/MICCAI2026, leave ZIP_NAME empty to auto-detect.
# If there are multiple zips, set it exactly, e.g. ZIP_NAME = "MICCAI2026.zip".
ZIP_NAME = "miccai.zip"
CLEAN_WORKDIR = True
print("Drive project:", DRIVE_PROJECT)
print("Workdir:", WORKDIR)
print("Run dir:", RUN_DIR)
print("Submission root:", SUBMISSION_ROOT)

In [ ]:
#@title Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
#@title Unpack Project Zip Or Copy Folders From Drive
import shutil
import zipfile
from pathlib import Path

WORKDIR.mkdir(parents=True, exist_ok=True)

if CLEAN_WORKDIR and WORKDIR.exists():
    for child in WORKDIR.iterdir():
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()

extract_dir = Path("/content/_miccai2026_zip_extract")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

zip_candidates = sorted(DRIVE_PROJECT.glob("*.zip"))
selected_zip = None
if ZIP_NAME:
    selected_zip = DRIVE_PROJECT / ZIP_NAME
    if not selected_zip.exists():
        raise FileNotFoundError(f"ZIP_NAME was set but not found: {selected_zip}")
elif zip_candidates:
    preferred = [
        DRIVE_PROJECT / "miccai.zip",
        DRIVE_PROJECT / "MICCAI2026.zip",
        DRIVE_PROJECT / "miccai2026.zip",
        DRIVE_PROJECT / "GAVE2_preliminary.zip",
        DRIVE_PROJECT / "gave2_preliminary.zip",
        DRIVE_PROJECT / "gave2_colab.zip",
    ]
    selected_zip = next((p for p in preferred if p.exists()), zip_candidates[0])


def copy_dir(src: Path, dst: Path) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"copied {src} -> {dst}")


def find_named_dir(root: Path, name: str):
    direct = root / name
    if direct.is_dir():
        return direct
    matches = [p for p in root.rglob(name) if p.is_dir()]
    return matches[0] if matches else None


def find_dataset_root(root: Path):
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for p in candidates:
        if (p / "training").is_dir() and (p / "validation").is_dir():
            return p
    return None

if selected_zip is not None:
    print("Using zip:", selected_zip)
    with zipfile.ZipFile(selected_zip) as zf:
        zf.extractall(extract_dir)

    dataset_src = find_named_dir(extract_dir, "GAVE2_preliminary") or find_dataset_root(extract_dir)
    if dataset_src is None:
        raise FileNotFoundError("Could not find GAVE2_preliminary or training/validation folders inside zip")
    copy_dir(dataset_src, WORKDIR / "GAVE2_preliminary")

    for name in ["experiments", "knowledge_base"]:
        src = find_named_dir(extract_dir, name)
        if src is not None:
            copy_dir(src, WORKDIR / name)
        elif (DRIVE_PROJECT / name).is_dir():
            copy_dir(DRIVE_PROJECT / name, WORKDIR / name)
        else:
            print(f"optional folder not found: {name}")
else:
    print("No zip found. Falling back to folder copy from Drive.")
    for name in ["GAVE2_preliminary", "experiments", "knowledge_base"]:
        src = DRIVE_PROJECT / name
        if src.is_dir():
            copy_dir(src, WORKDIR / name)
        elif name == "knowledge_base":
            print("optional folder not found: knowledge_base")
        else:
            raise FileNotFoundError(f"Required folder not found and no zip available: {src}")

if not (WORKDIR / "experiments" / "gave2_ensemble").is_dir():
    raise FileNotFoundError(
        "experiments/gave2_ensemble was not found. Include the experiments folder in the zip, "
        "or keep MyDrive/MICCAI2026/experiments as a Drive folder."
    )

print("Ready workspace:", WORKDIR)
print("Dataset:", WORKDIR / "GAVE2_preliminary")
print("Experiment code:", WORKDIR / "experiments" / "gave2_ensemble")

In [ ]:
#@title Install / Check Dependencies
%%bash
set -e
python -m pip install -q --upgrade pip
python -m pip install -q numpy pillow
python - <<'PY'
import sys
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM GB:", round(props.total_memory / 1024**3, 2))
PY

In [ ]:
#@title Verify Dataset Shape
import sys
sys.path.insert(0, str(WORKDIR))

from experiments.gave2_ensemble.data import GAVE2Dataset

train = GAVE2Dataset(DATA_ROOT, "training", "task2")
val = GAVE2Dataset(DATA_ROOT, "validation", "task2", require_target=False)
train_sample = train[0]
val_sample = val[0]

print("Training cases:", len(train), train_sample.case_id, train_sample.image.shape, train_sample.target.shape)
print("Validation cases:", len(val), val_sample.case_id, val_sample.image.shape, val_sample.mask.shape)
assert train_sample.image.shape == (5, 1024, 1536)
assert train_sample.target.shape == (3, 1024, 1536)
assert val_sample.image.shape == (5, 1024, 1536)

## 1. Smoke Test CMRRWNet

Run this first. It trains on only two cases for one epoch, just to check the environment and memory path.

In [ ]:
#@title Smoke Test: CMRRWNet Task 2
%%bash
set -e
cd /content/MICCAI2026
python -m experiments.gave2_ensemble.train \
  --branch cmrrwnet \
  --task task2 \
  --data-root GAVE2_preliminary \
  --out-dir /content/drive/MyDrive/MICCAI2026/runs/gave2_cmrrwnet_smoke \
  --epochs 1 \
  --folds 2 \
  --limit-cases 2 \
  --base-channels 16 \
  --batch-size 1 \
  --grad-accum 1 \
  --amp bf16 \
  --workers 2

## 2. Train CMRRWNet Task 2 First

Task 2 has 40% leaderboard weight and uses CFP + FFA. Run one fold at a time. After fold 0 finishes, change `FOLD = 1`, then `2`, `3`, `4`.

In [ ]:
#@title Train CMRRWNet Task 2: One Fold
FOLD = 0  # change to 1, 2, 3, 4 after each fold finishes

!cd {WORKDIR} && python -m experiments.gave2_ensemble.train \
  --branch cmrrwnet \
  --task task2 \
  --data-root {DATA_ROOT} \
  --out-dir {RUN_DIR} \
  --epochs {EPOCHS_FULL} \
  --folds {FOLDS} \
  --fold {FOLD} \
  --batch-size {BATCH_SIZE} \
  --grad-accum {GRAD_ACCUM} \
  --base-channels {BASE_CHANNELS} \
  --num-iterations 5 \
  --amp {AMP} \
  --workers {WORKERS} \
  --preprocess {PREPROCESS} \
  --loss-mode {LOSS_MODE} \
  --early-stopping-patience {EARLY_STOPPING_PATIENCE} \
  --early-stopping-min-delta {EARLY_STOPPING_MIN_DELTA} \
  --early-stopping-metric best_dice

## 3. Train CMRRWNet Task 1

After Task 2 folds are done, train Task 1 with CFP-only input. Again, run one fold at a time.

In [ ]:
#@title Train CMRRWNet Task 1: One Fold
FOLD = 0  # change to 1, 2, 3, 4 after each fold finishes

!cd {WORKDIR} && python -m experiments.gave2_ensemble.train \
  --branch cmrrwnet \
  --task task1 \
  --data-root {DATA_ROOT} \
  --out-dir {RUN_DIR} \
  --epochs {EPOCHS_FULL} \
  --folds {FOLDS} \
  --fold {FOLD} \
  --batch-size {BATCH_SIZE} \
  --grad-accum {GRAD_ACCUM} \
  --base-channels {BASE_CHANNELS} \
  --num-iterations 5 \
  --amp {AMP} \
  --workers {WORKERS} \
  --preprocess {PREPROCESS} \
  --loss-mode {LOSS_MODE} \
  --early-stopping-patience {EARLY_STOPPING_PATIENCE} \
  --early-stopping-min-delta {EARLY_STOPPING_MIN_DELTA} \
  --early-stopping-metric best_dice

In [ ]:
#@title Check CMRRWNet Checkpoints
for task in ["task2", "task1"]:
    ckpts = sorted((RUN_DIR / "cmrrwnet" / task).glob("fold_*/best.pt"))
    print(task, len(ckpts), "best checkpoints")
    for p in ckpts:
        print("  ", p)

## 4. Predict CMRRWNet Outputs

This creates Task 1 and Task 2 probability PNGs for validation cases.

In [ ]:
#@title Predict Task 1 And Task 2
for task in ["task1", "task2"]:
    !cd {WORKDIR} && python -m experiments.gave2_ensemble.predict \
      --branch cmrrwnet \
      --task {task} \
      --data-root {DATA_ROOT} \
      --run-dir {RUN_DIR} \
      --output-root {SUBMISSION_ROOT} \
      --team-id {TEAM_ID} \
      --tta flips \
      --preprocess auto \
      --workers {WORKERS}

## 5. Generate Task 3 For CMRRWNet

This creates biomarker TXT files from CMRRWNet Task 2 predictions.

Note: this is currently a deterministic proxy from Task 2 probability maps and ROI masks. It gives complete Task 3 files for testing, but the later leaderboard-focused improvement should add optic-disc-aware Zone C calibration.

In [ ]:
#@title Generate Task 3 TXT Files
!cd {WORKDIR} && python -m experiments.gave2_ensemble.biomarkers \
  --branch cmrrwnet \
  --data-root {DATA_ROOT} \
  --submission-root {SUBMISSION_ROOT} \
  --team-id {TEAM_ID}

## 6. Validate And Zip CMRRWNet Submission

The validator checks `50` Task 1 PNGs, `50` Task 2 PNGs, and `50` Task 3 TXT files, with exact `1536 x 1024` RGB probability PNGs.

In [ ]:
#@title Validate CMRRWNet Folder Only
from experiments.gave2_ensemble.data import list_case_ids
from experiments.gave2_ensemble.submission import validate_submission_tree

case_ids = list_case_ids(DATA_ROOT, split="validation")
report = validate_submission_tree(
    SUBMISSION_ROOT / "cmrrwnet" / TEAM_ID,
    case_ids,
    expected_size=(1024, 1536),
)
print("OK:", report.ok)
print("Counts:", report.counts)
print("Errors:", report.errors[:10])
assert report.ok

In [ ]:
#@title Zip CMRRWNet Output For Upload
ZIP_PATH = DRIVE_PROJECT / f"cmrrwnet_{TEAM_ID}.zip"
!cd {SUBMISSION_ROOT / "cmrrwnet"} && zip -qr {ZIP_PATH} {TEAM_ID}
print("Wrote", ZIP_PATH)

## Exercise

After the first successful CMRRWNet run, record:

```text
Task 2 folds completed: __/5
Task 1 folds completed: __/5
Validation report OK: yes/no
First leaderboard score: ________
```

Once we see this score, we can decide whether to improve CMRRWNet itself or move to SAM3/YOLO ensemble branches.

## Later: Ensemble Path

Do not run this notebook for SAM3/YOLO yet. After CMRRWNet is submitted/tested, use the separate four-output notebook:

```text
experiments/gave2_ensemble/GAVE2_Four_Output_Colab.ipynb
```

That notebook keeps CMRRWNet, SAM3-native, YOLO-native, and ensemble outputs separate.